# AI Programming — Lecture 23
## Lab 2: Pretrained ProGAN

이번 실습에서는 강의자료에 제시된 **pretrained ProGAN**을 그대로 활용하여
고해상도 이미지 생성을 간단히 경험합니다.

ProGAN 자체를 처음부터 학습하지 않습니다.

### 강의에서 배운 핵심

```text
Low Resolution
     ↓
Progressively Grow
     ↓
Higher Resolution
     ↓
High-Resolution Image
```

ProGAN의 주요 아이디어:

- Progressive Growing
- Fade-In
- Mini-Batch Standard Deviation

이번 Notebook에서는 구현보다
**pretrained ProGAN의 latent space를 사용한 image generation**에 집중합니다.

### 학습 목표

- TensorFlow Hub에서 pretrained ProGAN을 불러옵니다.
- Random latent vector에서 이미지를 생성합니다.
- Random seed에 따라 생성 결과가 달라짐을 확인합니다.
- 두 latent vector 사이를 interpolation합니다.

## 0. 실습 환경 설정

In [ ]:
!pip install -q tensorflow-hub

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_hub as hub

print("TensorFlow:", tf.__version__)
print("TensorFlow Hub:", hub.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus[0].name if gpus else "사용하지 않음 (CPU)")

## 1. Pretrained ProGAN 불러오기

강의자료의 코드를 그대로 사용합니다.

```python
module = hub.KerasLayer(
    'https://tfhub.dev/google/progan-128/1'
)

latent_dim = 512
```

Input은 512-dimensional latent vector이고,
output은 $128 \times 128 \times 3$ RGB image입니다.

In [ ]:
module = hub.KerasLayer(
    'https://tfhub.dev/google/progan-128/1'
)

latent_dim = 512

print("Latent dimension:", latent_dim)

## 2. 강의자료의 기본 예제

In [ ]:
latent_vector = tf.random.normal(
    [1, latent_dim],
    seed=123456789
)

generated_image = module(
    latent_vector
)

print(
    "Generated tensor shape:",
    generated_image.shape
)

plt.figure(figsize=(4, 4))

plt.imshow(
    generated_image
    .numpy()
    .reshape(128, 128, 3)
)

plt.axis("off")
plt.title("Pretrained ProGAN")
plt.show()

## 3. Seed를 바꾸어 이미지 생성

강의자료에서는 서로 다른 seed 예제를 보여줍니다.

```text
Seed 12345
Seed 11111
```

Seed가 달라지면 latent vector가 달라지고,
생성 결과도 달라집니다.

In [ ]:
seeds = [
    12345,
    11111,
    2026,
    77777,
]

plt.figure(figsize=(10, 3))

for i, seed in enumerate(seeds):
    z = tf.random.normal(
        [1, latent_dim],
        seed=seed
    )

    image = module(z)

    plt.subplot(
        1,
        len(seeds),
        i + 1
    )

    plt.imshow(
        image
        .numpy()
        .reshape(128, 128, 3)
    )

    plt.title(
        f"Seed {seed}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Latent Space Interpolation

두 latent vector $z_1$, $z_2$ 사이를 선형 보간합니다.

$$
z(\alpha)
=
(1-\alpha)z_1
+
\alpha z_2
$$

```text
z1 ───────────────────── z2
↓                         ↓
Image A → ... → ... → Image B
```

DCGAN에서도 확인했던 latent-space interpolation을
고해상도 pretrained GAN에서 다시 확인합니다.

In [ ]:
z_start = tf.random.normal(
    [1, latent_dim],
    seed=12345
)

z_end = tf.random.normal(
    [1, latent_dim],
    seed=11111
)

alphas = np.linspace(
    0.0,
    1.0,
    8
).astype("float32")

z_interp = tf.concat(
    [
        (1.0 - a) * z_start
        + a * z_end
        for a in alphas
    ],
    axis=0
)

interpolated_images = module(
    z_interp
)

print(
    "Interpolated images:",
    interpolated_images.shape
)

In [ ]:
plt.figure(figsize=(16, 3))

for i in range(len(alphas)):
    plt.subplot(
        1,
        len(alphas),
        i + 1
    )

    plt.imshow(
        interpolated_images[i]
        .numpy()
        .reshape(128, 128, 3)
    )

    plt.title(
        f"{alphas[i]:.2f}"
    )

    plt.axis("off")

plt.suptitle(
    "ProGAN Latent Space Interpolation"
)

plt.tight_layout()
plt.show()

## 5. DCGAN과 ProGAN 비교

### DCGAN

```text
MNIST
→ Generator / Discriminator 직접 구현
→ Alternating Training
→ GAN 학습 원리 이해
```

### ProGAN

```text
Pretrained model
→ Latent vector 입력
→ 128×128 RGB image
→ Latent interpolation
```

ProGAN 실습에서는 Progressive Growing 자체를 구현하지 않습니다.
강의에서 **고해상도 GAN training을 안정화하는 방법**으로 이해하는 것으로 충분합니다.

## 6. 직접 해보기

1. 여러 seed를 사용해 생성 이미지를 비교하세요.
2. interpolation step을 `8 → 16`으로 늘려 보세요.
3. 서로 다른 두 latent vector 사이를 interpolation해 보세요.
4. DCGAN의 latent interpolation과 ProGAN의 interpolation을 비교하세요.

### 생각해 보기

- Latent space에서 작은 이동이 이미지의 어떤 속성을 바꾸나요?
- 이미지가 interpolation 과정에서 갑자기 바뀌나요, 아니면 비교적 부드럽게 바뀌나요?
- Progressive Growing은 training 방법인데, pretrained inference에서는 왜 직접 보이지 않을까요?

# 정리

### ProGAN의 강의 핵심

```text
Low-resolution training
→ New layers added
→ Fade-in
→ Higher resolution
```

### 이번 실습

```text
z ∈ R^512
↓
Pretrained ProGAN
↓
128 × 128 RGB Image
```

### 꼭 기억할 것

1. **ProGAN은 고해상도 GAN training을 안정화하기 위해 progressive growing을 사용합니다.**
2. **새 resolution으로 넘어갈 때 fade-in을 사용합니다.**
3. **Mini-batch standard deviation은 sample diversity를 discriminator가 확인하도록 돕습니다.**
4. **이번 실습에서는 pretrained model을 사용하므로 ProGAN training 자체는 수행하지 않습니다.**
5. **Latent interpolation을 통해 latent space의 연속적인 구조를 관찰할 수 있습니다.**